In [1]:
include("../src/TensorDecomposition.jl")
using LinearAlgebra, LinearSolve


Welcome to Nemo version 0.49.5

Nemo comes with absolutely no warranty whatsoever


In [2]:
n = 3
r = 7

D, Drev = TensorDecomposition.makeDicts(n, 4);
basis_inds = collect(1:r)
basis, basisD = TensorDecomposition.basisFn(basis_inds, Drev);

vars = TensorDecomposition.varTups(basis, n, 4)
eqs1, eqs2 = TensorDecomposition.linEqTups(basisD, n, 4);

r

7

In [15]:
Z = rand(-5:5, n+1, r)
T = TensorDecomposition.rankedTensor(ones(r), Z, 4; type=eltype(Z));

Tcat = TensorDecomposition.catMat(T, 2)
H0 = Tcat[basis_inds, basis_inds]

7×7 Matrix{Float64}:
 1860.0     7.0   548.0  -56.0  625.0  -188.0  438.0
    7.0   625.0  -188.0  438.0  247.0    84.0  160.0
  548.0  -188.0   730.0  -35.0   84.0  -268.0    6.0
  -56.0   438.0   -35.0  982.0  160.0     6.0  -78.0
  625.0   247.0    84.0  160.0  386.0   -45.0   92.0
 -188.0    84.0  -268.0    6.0  -45.0   133.0   12.0
  438.0   160.0     6.0  -78.0   92.0    12.0  420.0

In [16]:
Z

4×7 Matrix{Int64}:
  5  5  -2  -3   4   1  -4
 -2  4   0   2  -1   3   2
  1  0  -4  -5   5   0   1
 -5  3  -2  -2   0  -4  -2

In [20]:
A, b = TensorDecomposition.linearSystem(T, H0, basis_inds, basisD, D, vars, eqs1, eqs2; type=eltype(T));

A_ = Matrix(copy(A))
foreach(TensorDecomposition.normalize!, eachcol(A_));
foreach(TensorDecomposition.normalize!, eachrow(A_));
svdvals(A_)

10-element Vector{Float64}:
 1.9560459197718663
 1.5377934411761283
 1.4979289793673851
 1.3583282930544935
 1.092433256662851
 0.5480607103177402
 0.35881535137548076
 0.2543805451013002
 0.1812331047231809
 0.01201641648863476

In [21]:
prob = LinearProblem(A, b)
sol = solve(prob)
solDict = Dict([v => s for (v, s) in zip(vars, sol.u)]);
Ms = TensorDecomposition.multMatrices(T, basis, solDict, D, H0)
lhat, Zhat = TensorDecomposition.obtainDecomp(T, Ms);

In [22]:
maximum(abs.(T-TensorDecomposition.rankedTensor(lhat, Zhat, 4, type=eltype(Zhat))))/maximum(abs.(T))

9.192742146952178e-14